# Load Python packages

In [1]:
#Load the libraries
import numpy as np
import pandas as pd
import yfinance as yf
from scipy.signal import argrelextrema

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Load OHLC prices (NVDA)

In [2]:
ticker_symbol = 'NVDA'
start_date = '2012-01-01'

In [3]:
def LoadPricesFromYF(symbol, start_date):
    stock_df = pd.DataFrame([], columns = [])
        
    # print(f"CSV file of {symbol} prices aren't stored")
    stock_prices = yf.Ticker(symbol).history(start=start_date, auto_adjust=False)
    stock_df['Open'] = stock_prices['Open']
    stock_df['High'] = stock_prices['High']
    stock_df['Low'] = stock_prices['Low']
    stock_df['Close'] = stock_prices['Close']
    stock_df['Adj Close'] = stock_prices['Adj Close']
    stock_df['Volume'] = stock_prices['Volume']
    return stock_df

In [4]:
df = LoadPricesFromYF(ticker_symbol, start_date)

# Harmonic Pattern Detection

Detects classic harmonic patterns (Gartley, Butterfly, Bat, Crab) using Fibonacci ratio validation on XABCD swing points.

In [5]:
# ============================================================
# HARMONIC DETECTION CONFIGURATION — Adjust these to tune sensitivity
# ============================================================

# --- Swing Point Detection ---
# swing_orders: list of `order` values for argrelextrema when finding
#   swing highs/lows. Each value produces a different set of swings;
#   all are scanned for harmonic patterns.
#   Lower order = more swing points = more XABCD candidates.
#   order=3 catches very minor swings; order=15 catches only major pivots.
HARMONIC_SWING_ORDERS = [3, 5, 8, 12]

# --- Fibonacci Ratio Tolerance ---
# global_tolerance_pct: extra percentage tolerance added to ALL ratio checks.
#   Each harmonic pattern has ideal Fibonacci ratios (e.g. Gartley XB = 0.618).
#   The base tolerance is pattern-specific (defined in HARMONIC_RATIOS below).
#   This global value is ADDED on top, uniformly relaxing all checks.
#   0.0 = use only base tolerance   0.10 = add 10% extra flexibility
#   0.20 = very relaxed (may produce false positives)
HARMONIC_GLOBAL_TOLERANCE_PCT = 0.15

# --- Ratio Definitions ---
# Each entry defines the expected Fibonacci ratios for a harmonic pattern.
# Format:
#   Single target: (target_ratio, base_tolerance)
#     Passes if |actual - target| <= base_tolerance + target * GLOBAL_TOLERANCE_PCT
#   Range target:  (low_ratio, high_ratio, base_tolerance)
#     Passes if (low - tol - low*GLOBAL_TOL) <= actual <= (high + tol + high*GLOBAL_TOL)
#
# The four ratios:
#   XB = AB / XA : how far B retraces the XA leg
#   AC = BC / AB : how far C retraces the AB leg
#   BD = CD / BC : extension of the CD leg relative to BC
#   XD = AD / XA : how far D retraces (or extends) the full XA leg
#
# Increase base_tolerance values to match more patterns at the cost of precision.
HARMONIC_RATIOS = {
    'Gartley': {
        # XB ≈ 0.618 (B retraces 61.8% of XA)
        'XB': (0.618, 0.08),
        # AC between 0.382 and 0.886 (C retraces 38.2%-88.6% of AB)
        'AC': (0.382, 0.886, 0.08),
        # BD between 1.27 and 1.618 (CD extends 127%-161.8% of BC)
        'BD': (1.27, 1.618, 0.08),
        # XD ≈ 0.786 (D completes near 78.6% retracement of XA)
        'XD': (0.786, 0.10),
    },
    'Butterfly': {
        # XB ≈ 0.786
        'XB': (0.786, 0.08),
        # AC between 0.382 and 0.886
        'AC': (0.382, 0.886, 0.08),
        # BD between 1.618 and 2.618 (large CD extension)
        'BD': (1.618, 2.618, 0.10),
        # XD ≈ 1.27 (D extends beyond X)
        'XD': (1.27, 0.12),
    },
    'Bat': {
        # XB between 0.382 and 0.50
        'XB': (0.382, 0.50, 0.08),
        # AC between 0.382 and 0.886
        'AC': (0.382, 0.886, 0.08),
        # BD between 1.618 and 2.618
        'BD': (1.618, 2.618, 0.10),
        # XD ≈ 0.886
        'XD': (0.886, 0.10),
    },
    'Crab': {
        # XB between 0.382 and 0.618
        'XB': (0.382, 0.618, 0.08),
        # AC between 0.382 and 0.886
        'AC': (0.382, 0.886, 0.08),
        # BD between 2.24 and 3.618 (very large CD extension)
        'BD': (2.24, 3.618, 0.12),
        # XD ≈ 1.618 (D extends well beyond X)
        'XD': (1.618, 0.12),
    },
}

# --- Deduplication ---
# dedup_distance: minimum index distance between pattern D-points to
#   consider them distinct patterns. Patterns with D-points closer than
#   this are merged (first found wins).
HARMONIC_DEDUP_DISTANCE = 5

In [7]:
def ratio_check(actual, expected, global_tol_pct=0.15):
    """
    Check if an actual Fibonacci ratio matches the expected value(s).

    Parameters
    ----------
    actual : float
        The computed ratio from the price data (e.g. AB/XA).
    expected : tuple
        Either (target, base_tol) for a single-value check, or
        (low, high, base_tol) for a range check.
    global_tol_pct : float
        Extra tolerance as a fraction of the target/range bounds.
        Added on top of base_tol to relax matching.

    Returns
    -------
    bool : True if actual falls within the acceptable range.

    How it works
    ------------
    - Single target (target, tol): passes if
        |actual - target| <= tol + target * global_tol_pct
      The effective window is [target - effective_tol, target + effective_tol].

    - Range (lo, hi, tol): passes if
        (lo - tol - lo*global_tol) <= actual <= (hi + tol + hi*global_tol)
      This widens both ends of the range.
    """
    if len(expected) == 2:
        target, tol = expected
        effective_tol = tol + target * global_tol_pct
        return abs(actual - target) <= effective_tol
    else:
        lo, hi, tol = expected
        return (lo - tol - lo * global_tol_pct) <= actual <= (hi + tol + hi * global_tol_pct)

def find_swing_points(close, order=10):
    """
    Find alternating swing highs (H) and lows (L) in the price series.

    Parameters
    ----------
    close : np.ndarray
        1D array of closing prices.
    order : int
        Number of bars on each side that a point must exceed to be an extremum.
        Lower = more sensitive, more swing points found.

    Returns
    -------
    list[tuple] : [(index, 'H'|'L', price), ...] in chronological order,
        guaranteed to alternate between H and L. When consecutive same-type
        points occur, only the most extreme one is kept.
    """
    highs = argrelextrema(close, np.greater, order=order)[0]
    lows = argrelextrema(close, np.less, order=order)[0]

    # Merge all swing points and sort chronologically
    all_points = [(i, 'H', close[i]) for i in highs] + [(i, 'L', close[i]) for i in lows]
    all_points.sort(key=lambda x: x[0])

    # Enforce strict alternation (H, L, H, L, ...)
    # If two consecutive highs, keep only the higher one; same for lows.
    filtered = [all_points[0]] if all_points else []
    for p in all_points[1:]:
        if p[1] != filtered[-1][1]:
            filtered.append(p)
        elif (p[1] == 'H' and p[2] > filtered[-1][2]) or (p[1] == 'L' and p[2] < filtered[-1][2]):
            filtered[-1] = p

    return filtered

def detect_harmonics(
    df,
    swing_orders=None,
    harmonic_ratios=None,
    global_tol_pct=0.15,
    dedup_distance=5,
):
    """
    Scan for XABCD harmonic patterns in price data.

    Parameters
    ----------
    df : pd.DataFrame
        OHLCV DataFrame with DatetimeIndex.
    swing_orders : list[int]
        List of `order` values for swing point detection. Each order produces
        a different set of XABCD candidates. More orders = broader scan.
        Default: [3, 5, 8, 12].
    harmonic_ratios : dict
        Dictionary of pattern names to Fibonacci ratio specifications.
        See HARMONIC_RATIOS configuration for format details.
    global_tol_pct : float
        Global tolerance percentage added to all ratio checks.
        Higher = more permissive matching. Default: 0.15.
    dedup_distance : int
        Minimum index distance between D-points for patterns to be
        considered distinct. Default: 5.

    Returns
    -------
    list[dict] : Detected harmonic patterns with type, XABCD points,
        computed ratios, direction, and entry price.

    Key Variables in the Algorithm
    ------------------------------
    - swings : alternating sequence of swing highs and lows
    - X, A, B, C, D : five consecutive swing points forming the XABCD structure
    - xp, ap, bp, cp, dp : price values at each point
    - xa : |A - X| price magnitude of the initial XA impulse leg
    - ab : |B - A| retracement of XA (the AB leg)
    - bc : |C - B| retracement of AB (the BC leg)
    - cd : |D - C| extension from C (the CD leg)
    - xb_ratio = ab/xa : how much B retraces XA (key Fib level)
    - ac_ratio = bc/ab : how much C retraces AB
    - bd_ratio = cd/bc : how much D extends from BC
    - xd_ratio = |D-X|/xa : overall retracement/extension of D vs X
    - direction : 'Bullish' if D is below X (expect reversal up),
                  'Bearish' if D is above X (expect reversal down)
    """
    if swing_orders is None:
        swing_orders = [3, 5, 8, 12]
    if harmonic_ratios is None:
        harmonic_ratios = HARMONIC_RATIOS

    close = df['Close'].values.flatten()
    all_patterns = []

    # Scan with multiple swing-detection sensitivities
    for order in swing_orders:
        swings = find_swing_points(close, order=order)

        # Slide a 5-point window across the swing sequence
        for i in range(len(swings) - 4):
            X, A, B, C, D = swings[i:i + 5]

            # Extract prices at each swing point
            xp, ap, bp, cp, dp = X[2], A[2], B[2], C[2], D[2]

            # XA leg magnitude — the reference move all ratios are based on
            xa = abs(ap - xp)
            if xa < 1e-6:
                continue  # degenerate, skip

            # Leg magnitudes
            ab = abs(bp - ap)  # AB retracement of XA
            bc = abs(cp - bp)  # BC retracement of AB
            cd = abs(dp - cp)  # CD extension from C

            # Compute the four defining Fibonacci ratios
            xb_ratio = ab / xa                          # B's retracement of XA
            ac_ratio = bc / ab if ab > 1e-6 else 999    # C's retracement of AB
            bd_ratio = cd / bc if bc > 1e-6 else 999    # D's extension from BC
            xd_ratio = abs(dp - xp) / xa                # D's position relative to X

            # Check against each harmonic pattern's ideal ratios
            for name, ratios in harmonic_ratios.items():
                if (ratio_check(xb_ratio, ratios['XB'], global_tol_pct) and
                    ratio_check(ac_ratio, ratios['AC'], global_tol_pct) and
                    ratio_check(bd_ratio, ratios['BD'], global_tol_pct) and
                    ratio_check(xd_ratio, ratios['XD'], global_tol_pct)):

                    all_patterns.append({
                        'type': name,
                        'swing_order': order,
                        'points': {'X': X, 'A': A, 'B': B, 'C': C, 'D': D},
                        'ratios': {'XB': xb_ratio, 'AC': ac_ratio, 'BD': bd_ratio, 'XD': xd_ratio},
                        'direction': 'Bullish' if dp < xp else 'Bearish',
                        'completion_date': df.iloc[D[0]].name,
                        'entry_price': dp,
                    })

    # Deduplicate across all swing orders
    unique = []
    for h in all_patterns:
        d_idx = h['points']['D'][0]
        if not any(abs(d_idx - u['points']['D'][0]) < dedup_distance and
                   h['type'] == u['type'] for u in unique):
            unique.append(h)

    return unique


# --- Run detection ---
unique_harmonics = detect_harmonics(
    df,
    swing_orders=HARMONIC_SWING_ORDERS,
    harmonic_ratios=HARMONIC_RATIOS,
    global_tol_pct=HARMONIC_GLOBAL_TOLERANCE_PCT,
    dedup_distance=HARMONIC_DEDUP_DISTANCE,
)

# --- Results ---
print(f"Found {len(unique_harmonics)} harmonic pattern(s):\n")
for i, h in enumerate(unique_harmonics):
    print(f"  [{i+1}] {h['direction']} {h['type']}  (swing_order={h['swing_order']})")
    print(f"      Completion: {h['completion_date'].strftime('%Y-%m-%d')}")
    print(f"      Entry Price: ${h['entry_price']:.2f}")
    print(f"      Ratios — XB: {h['ratios']['XB']:.3f}, AC: {h['ratios']['AC']:.3f}, BD: {h['ratios']['BD']:.3f}, XD: {h['ratios']['XD']:.3f}")
    print()

Found 3 harmonic pattern(s):

  [1] Bearish Butterfly  (swing_order=5)
      Completion: 2015-02-17
      Entry Price: $0.56
      Ratios — XB: 0.957, AC: 0.962, BD: 2.106, XD: 0.976

  [2] Bullish Butterfly  (swing_order=8)
      Completion: 2022-10-05
      Entry Price: $13.21
      Ratios — XB: 0.926, AC: 0.454, BD: 1.821, XD: 1.259

  [3] Bearish Crab  (swing_order=12)
      Completion: 2013-05-21
      Entry Price: $0.37
      Ratios — XB: 0.718, AC: 0.821, BD: 4.043, XD: 1.513



# Plot Harmonic Patterns

In [20]:
# --- Plot top harmonic patterns ---
for pat in unique_harmonics[:6]:
    pts = pat['points']

    x_idx = pts['X'][0]
    d_idx = pts['D'][0]
    margin = 20
    start = max(0, x_idx - margin)
    end = min(len(df), d_idx + margin)
    seg = df.iloc[start:end]

    # ax.plot(segment.index, segment['Close'], color='#90CAF9', linewidth=1, alpha=0.8)
    # Plot candlestick chart
    fig = go.Figure(data=[go.Candlestick(x=seg.index,
                open=seg['Open'],
                high=seg['High'],
                low=seg['Low'],
                close=seg['Close'])])

    labels = ['X', 'A', 'B', 'C', 'D']
    coords = [(df.index[pts[l][0]], pts[l][2]) for l in labels]
    xs, ys = zip(*coords)
    fig.add_trace(go.Scatter(
    x=xs,
    y=ys,
    mode="lines",
    line=dict(color='#FF9800', width=2),
    marker=dict(symbol='circle', size=10, color='#FF9800'),
    name="XABCD Points"
    ))
    for label, (x, y) in zip(labels, zip(xs, ys)):
        fig.add_annotation(
            x=x,
            y=y,
            text=label,
            yshift=15,
            showarrow=False,
            font=dict(size=14, color='#FF9800')
        )

    if pat['type'] is not None:
        color = '#4CAF50' if pat['direction'] == 'Bullish' else '#F44336' if pat['direction'] == 'Bearish' else '#FF9800'
    else:
        color = '#FF9800'  # Default orange for pattern name
    fig.update_layout(
        title=f"Candles {start} → {end} (pivots + XABCD Points)\n" + 
        (f"{pat['direction']} {pat['type']} — Entry: ${pat['entry_price']:.2f}  (order={pat['swing_order']})"),
        xaxis_title="Index",
        yaxis_title="Price",
        xaxis_rangeslider_visible=False,
        template="plotly_dark",
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
        height=650, width=1100,
        title_font_color = color
    )
    fig.show()

if not unique_harmonics:
    print("No harmonic patterns detected. Try increasing HARMONIC_GLOBAL_TOLERANCE_PCT or adding smaller HARMONIC_SWING_ORDERS.")